# Llama 3.2 1B, running locally via Ollama

The Ollama server runs as a background service on `http://localhost:11434` and holds the
weights; this notebook is just a client. Nothing here downloads or trains anything.

| | |
| --- | --- |
| Model | `llama3.2:1b` — 1.24B params, Q4_K_M quant, ~1.3 GB on disk |
| Context | 128K max, but `num_ctx` below caps what we actually allocate |
| Speed | ~41 tok/s on this M1 |

Kernel: `llms/.venv`. Server not responding? See section 0.


## 0. Managing the server

Yes — it runs in the background. `brew services` installs it as a launchd job, so it starts
on login and keeps running after you close this notebook and after a reboot. It idles at
near-zero CPU when nothing is calling it; the 1.3 GB of weights are only held in RAM for 5
minutes after the last request (see section 8).

These are shell commands, not Python. Run them in a terminal — or from a cell by prefixing
with `!`.

**One-time install**

```bash
brew install ollama            # CLI + server
ollama pull llama3.2:1b        # ~1.3 GB, stored in ~/.ollama/models
```

**Start**

```bash
brew services start ollama     # background, restarts at login  ← what's set up now
```

Or, if you'd rather not have a permanent service, run it in a terminal window and watch the
logs. It stops when you close the window or press Ctrl-C:

```bash
ollama serve
```

Use one or the other. Starting `ollama serve` while the service is already running fails
with `address already in use` — port 11434 is taken by the copy that's already up.

**Check**

```bash
curl http://localhost:11434    # -> Ollama is running
brew services list | grep ollama
ollama ps                      # which models are resident in RAM right now
```

**Stop**

```bash
brew services stop ollama      # stops it and cancels the start-at-login
brew services restart ollama   # after an upgrade, or if it gets wedged
```

Stopping only shuts down the process — your downloaded models stay on disk, and a later
`start` picks up where you left off. To actually reclaim the disk space:

```bash
ollama rm llama3.2:1b
```


## 1. Is the server up?

Run this first. Every other cell fails with a connection error if it doesn't pass.

In [11]:
import ollama

MODEL = "llama3.2:1b"

for m in ollama.list()["models"]:
    print(f"{m['model']:<20} {m['size'] / 1e9:.2f} GB")

llama3.2:1b          1.32 GB


## 2. One-shot chat

`chat()` wraps your text in Llama 3.2's instruction template before it reaches the model —
that's the difference between this and `generate()` in section 4.

In [12]:
resp = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "What is a transformer block? Two sentences."}],
    options={"temperature": 0.7, "num_ctx": 4096},  # num_ctx costs RAM; 8 GB machine, keep it small
)

print(resp["message"]["content"])
print("\n---")
print(f"prompt tokens: {resp['prompt_eval_count']}, generated: {resp['eval_count']}")
print(f"speed: {resp['eval_count'] / (resp['eval_duration'] / 1e9):.1f} tok/s")

A transformer block, also known as a transformer core, is a crucial component in electrical power systems, designed to transfer electrical energy from one circuit to another through electromagnetic induction. It consists of a set of coils of wire wrapped around a central iron core, which generates a magnetic field that facilitates the transfer of energy.

---
prompt tokens: 34, generated: 63
speed: 33.2 tok/s


## 3. Streaming

Same call, tokens printed as they are sampled instead of after the full response. This is
the autoregressive loop you build by hand in the tutorial's notebook 3 — one token, appended
to the context, then the next.

In [13]:
for chunk in ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a haiku about backpropagation."}],
    stream=True,
):
    print(chunk["message"]["content"], end="", flush=True)

Data flows back
Error weights the bias
Learning's gentle creep

## 4. Raw completion — no chat template

`generate()` feeds your string to the model verbatim. No system prompt, no instruction
wrapper: it just continues the text. This is the apples-to-apples comparison against your
own from-scratch model, which is also a bare next-token predictor.

In [14]:
out = ollama.generate(
    model=MODEL,
    prompt="Once upon a time in a kingdom by the sea,",
    options={"temperature": 0.8, "num_predict": 80},  # num_predict = max new tokens
)
print(out["response"])

Where the sun dipped into the ocean so blue,


## 5. What temperature actually does

Same prompt, same seed, three temperatures. `0.0` is greedy — always the argmax token, so
it's reproducible. Higher values flatten the distribution before sampling.

In [15]:
prompt = "In three words, describe the ocean:"

for temp in (0.0, 0.8, 1.5):
    out = ollama.generate(
        model=MODEL,
        prompt=prompt,
        options={"temperature": temp, "seed": 42, "num_predict": 20},
    )
    print(f"temp={temp}: {out['response'].strip()}")

temp=0.0: Vast and powerful.
temp=0.8: Majestic vastness.
temp=1.5: Global vast expanse.


## 6. Multi-turn conversation

Ollama is stateless between calls — it keeps no history for you. Memory *is* the list you
resend every turn, which is why long conversations get slower and eventually hit `num_ctx`.

In [16]:
history = [{"role": "system", "content": "You are terse. Answer in one short sentence."}]


def ask(question: str) -> str:
    history.append({"role": "user", "content": question})
    reply = ollama.chat(model=MODEL, messages=history)["message"]
    history.append(reply)  # the reply must go back in, or turn 2 has no idea what it said
    return reply["content"]


print(ask("Name one famous transformer paper."))
print(ask("Who wrote it?"))  # only resolvable if the history above was sent
print(f"\nmessages in context: {len(history)}")

"Metal and Plastic: A Study on the Frictional Properties of Transformed Materials"
"Metal and Plastic: A Study on the Frictional Properties of Transformed Materials" was written by Ivar Løvland and Oskar Eriksen.

messages in context: 5


## 7. Forcing structured output

`format=` constrains sampling to tokens that keep the output valid against your schema, so
the result parses every time — no "please respond only in JSON" pleading. Useful if you want
to generate instruction/response pairs for the fine-tuning dataset in tutorial notebook 5.

In [17]:
import json

schema = {
    "type": "object",
    "properties": {
        "instruction": {"type": "string"},
        "response": {"type": "string"},
    },
    "required": ["instruction", "response"],
}

out = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Invent one Q&A pair about neural networks."}],
    format=schema,
    options={"temperature": 0},
)

pair = json.loads(out["message"]["content"])
print(json.dumps(pair, indent=2))

{
  "instruction": "Here is a Q&A pair about neural networks:",
  "response": "What is the primary function of the 'activation function' in a neural network?"
}


## 8. Memory control

The model stays loaded in RAM for 5 minutes after the last call. On 8 GB that competes with
a PyTorch MPS training run, so unload it explicitly before switching to the tutorial
notebooks.

In [18]:
ollama.generate(model=MODEL, prompt="", keep_alive=0)  # unload now

loaded = ollama.ps()["models"]
print(loaded if loaded else "nothing resident — RAM is free")

nothing resident — RAM is free
